# Create possible connections -> option file

First, all municipalities that will disappear between 2000 and 2025 because lifted and merged with another will be connected through only one connection
if the municipality is an old one and unclosed in 2025 (existing in 2000 and 
2025), connection is forced to its biggest... 
if connected to a new municipality or renamed, it will be treated like all 
the others, because the new one will inherit the old municipalities properties

In [ ]:
%pip install -e ../../../
%pip install scipy
%pip install geopy
%pip install plotly
%pip install nbformat
%pip install pygmo

In [ ]:
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12811)
import pandas as pd

from water_futures_battle.core import Settings
from water_futures_battle.jurisdictions import build_state
from water_futures_battle.jurisdictions.entities import Province
from water_futures_battle.sources import build_sources
from water_futures_battle.pumping_stations import build_pumping_infrastructure

from epanet_utils import build_epyt

DEBUG_MODE = True
settings_desc = {
    "start_year": 2000,
    "end_year": 2024,
    "national_budget": 0,
    "lifeline_volume": 80,
    "seed": 128
}
settings = Settings.from_config(settings_desc)
nl_desc = {
    'name': 'Netherlands',
    'id': 'NL0000',
    'jurisdictions-static_properties': 'jurisdictions-static_properties.xlsx',
    'municipalities-dynamic_properties': 'municipalities-dynamic_properties.xlsx'
}
netherlands = build_state(
    nl_desc,
    data_path='../../jurisdictions/',
    settings=settings
    )

# Upload the demands generated in the `jurisdicitons-view_demands-historical` notebook
pvs_max_daily_dem_df = pd.read_csv(
    '../../jurisdictions/provinces-max_daily_demands-CMD.csv',
    index_col='year'
)
juris_max_hourly_dem_df = pd.read_csv(
    '../../jurisdictions/jurisdictions-max_hourly_demands-CMH.csv',
    index_col='year',
    parse_dates=True
)
juris_max_hourly_dem_df.index = juris_max_hourly_dem_df.index.year

sources_desc = {
    'sources-static_properties': 'sources-static_properties.xlsx',
    'groundwater-dynamic_properties': 'groundwater-dynamic_properties.xlsx',
    'surface_water-dynamic_properties': 'surface_water-dynamic_properties.xlsx',
    'desalination-dynamic_properties': 'desalination-dynamic_properties.xlsx',
    'groundwater-construction_size-max_permit_surplus': 0.3,
    'surface_water-construction_size-min': 0,
    'surface_water-construction_size-max': 0,
    'desalination-construction_size-min': 0,
    'desalination-construction_size-max': 0,
}
all_sources, sources_settings = build_sources(
    sources_desc,
    a_state=netherlands,
    data_path='../../sources/',
    settings=settings
)
sources_lookup = pd.read_excel(
    '../../sources/sources-static_properties.xlsx',
    sheet_name=list(all_sources.types_names),
    index_col='source_id'
)

provinces_peak2source_factor = { }
for province in netherlands.provinces:
    peak_demands__CMH = juris_max_hourly_dem_df[province.cbs_id]
    
    province_sources = all_sources.source_by_province(province)
    capacities__CMH = []
    for year in peak_demands__CMH.index:
        active_sources = [s for s in province_sources if s.is_active(when=year)]

        capacities__CMH.append(sum([s.nominal_capacity/24 for s in active_sources]))

    ratios = peak_demands__CMH.values / np.array(capacities__CMH)
    
    provinces_peak2source_factor[province] = max(ratios.max(), 1.25)

pumping_desc = {
    'pumping_stations-static_properties': 'pumping_stations/pumping_stations-static_properties.xlsx',
    'pump_options-static_properties': 'pumps/pump_options-static_properties.xlsx',
    'pump_options-dynamic_properties': 'pumps/pump_options-dynamic_properties.xlsx'

}
pump_options, pumping_stations = build_pumping_infrastructure(
    desc=pumping_desc,
    sources=all_sources,
    data_path='../../',
    settings=settings
)

pipe_options = pd.read_excel(
    '../../pipes/pipe_options-static_properties.xlsx',
    index_col='option_id'
)
BIGGEST_PIPE_OPTION = str(pipe_options['diameter'].idxmax())
UNDECIDED_PIPE_OPTION = "UNDECIDED"

def connection_id_gen(t: str, x: int) -> str:
    return f"C{t[0].upper()}{x:04d}"

In [ ]:
import define_cnns

# We need to count connections at national level, so that when we merge the individual
# networks there is no conflict. We start from 1 like for municipalities and sources.
# We define one for intraprovincial connections and one for source to muncipality connections
M2M = 'gemeente' # type for id
intrapv_cnns_dfs = [] # list for the provincial dataframes
n_nl_intrapv_cnns = 1 # national counter for this type of connection
S2M = 'source'
sources_cnns_dfs = []
n_nl_sources_cnns = 1

YEARS = range(2000, 2025)

for province in sorted(netherlands.provinces, key=lambda pv: pv.cbs_id):

    # ------- 1.A  
    temp_df, handled = define_cnns.lifted_to_existing(
        a_province=province,
        peak_demands_df=juris_max_hourly_dem_df
    )
    temp_df['connection_id'] = [
        connection_id_gen(t=M2M, x=i)
        for i in range(
            n_nl_intrapv_cnns,
            n_nl_intrapv_cnns+len(temp_df)
        )
    ]
    
    utility_options_df = temp_df.set_index('connection_id')
    n_nl_intrapv_cnns += len(temp_df)

    # ------- 1.B
    temp_df = define_cnns.sources(
        a_province=province,
        nl_sources=all_sources,
        nl_sources_lookups=sources_lookup
    )
    temp_df['connection_id'] = [
        connection_id_gen(t=S2M, x=i)
        for i in range(
            n_nl_sources_cnns,
            n_nl_sources_cnns+len(temp_df)
        )
    ]

    temp_df.set_index('connection_id', inplace=True)
    sources_cnns_dfs.append(temp_df)
    n_nl_sources_cnns += len(temp_df)

    # CHECKPOINT 1 -------
    if DEBUG_MODE:
        mixed_cnns_df = pd.concat([utility_options_df, sources_cnns_dfs[-1]], axis=0) if not utility_options_df.empty else sources_cnns_dfs[-1]
        build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[2000].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=2000,
            connections_df=mixed_cnns_df,
            dates_as_str=False,
            pipe_options_df=pipe_options,
            pipes_roughnesses_history=None,
            situation_name='checkpoint_1'
        ).close()
        build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[2024].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=2024,
            connections_df=mixed_cnns_df,
            dates_as_str=False,
            pipe_options_df=pipe_options,
            situation_name='checkpoint_1',
            pipes_roughnesses_history=None,
            future_sources=True
        ).close()

    # ------- 2
    temp_df = define_cnns.full_delaunay(
        a_province=province,
        handled_municipalities=handled,
        year=2000,
        default_option_id=UNDECIDED_PIPE_OPTION
    )
    temp_df['connection_id'] = [
        connection_id_gen(t=M2M, x=i)
        for i in range(
            n_nl_intrapv_cnns,
            n_nl_intrapv_cnns+len(temp_df)
        )
    ]

    if not temp_df.empty:
        temp_df.set_index('connection_id', inplace=True)
        if not utility_options_df.empty:
            utility_options_df = pd.concat([utility_options_df, temp_df], axis=0)
        else:
            utility_options_df = temp_df
        
        n_nl_intrapv_cnns += len(temp_df)

    if DEBUG_MODE:
        mixed_cnns_df = pd.concat([utility_options_df, sources_cnns_dfs[-1]], axis=0) if not utility_options_df.empty else sources_cnns_dfs[-1]
        build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[2000].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=2000,
            connections_df=mixed_cnns_df,
            dates_as_str=False,
            pipe_options_df=pipe_options,
            pipes_roughnesses_history=None,
            situation_name='checkpoint_2'
        ).close()

    utility_options_df['province'] = province.cbs_id
    sources_cnns_dfs[-1]['province'] = province.cbs_id

    # Now put in the global dataframe
    intrapv_cnns_dfs.append(utility_options_df)

# Merge back together
intrapv_cnns_df = pd.concat(intrapv_cnns_dfs, axis=0, join='outer')
sources_cnns_df = pd.concat(sources_cnns_dfs, axis=0, join='outer')

### Manual adjustments of sources and closing municipalities connections

After experimenting with the final networks for a bit, we came back and adjusted manually some connections pipes.

In [ ]:
# Zeeland, CS0164 is suuuuper long, thus we need even slower velocities.
# The heuristic puts PI08 (1250 mm), we increase to PI09 (1500) as it basically halves the headloss
sources_cnns_df.loc['CS0164', 'pipes-option_ids'] = 'PI09'

## Modify the municipalities connections

We operate on the networks in 2000 because all the changes will propagate from there. 

We need to remove impossible connections, disable and create the topology with some heuristic 
and finally to the remaining pipes, we add an age that makes the intended average around 50 years.



### Manually disable impossible connections

Because they are too long and could lead to weird behaviour/solutions as well as because they would not make sense or they are crossing other provinces.
We do it manually because there is no real logic behind it, we can do this just by looking at checkpoint 2 files.

In [ ]:
DISABLE_CNNS_IDX = [
    # Drenthe
    164, 163, 162, 161, 160, 158,
    # Flevoland
    268,
    # Frieslan
    116, 124, 133, 134, 135, 136, 137,
    # Gelderland
    459, 457, 452, 446, 432, 439, 431, 456, 453, 448, 455, 450, 445, 458, 454, 
    451, 447,
    # Groningen
    59, 58, 57, 56,
    # Limburg
    1321, 1319, 1313, 1314, 1320, 1318, 1316, 1312, 1309, 1305, 1300, 1317, 1315,
    1311, 1306, 1308,
    # N-brabant
    1182, 1181, 1180, 1178, 1179, 1176, 1177, 1174, 1172, 1170, 1173, 1175, 1165,
    # N-Holland
    717,716, 711, 715, 710, 714, 713, 709, 694, 671, 712, 702, 697, 705, 703, 
    692, 707, 700, 674,
    # Overijssel
    252, 253, 254, 255, 256, 257, 258,
    # Utrecht
    547, 552, 548, 538, 551, 540, 539, 553, 550, 534, 549, 545, 
    # Zeeland
    994, 995, 992, 988, 987, 993, 989, 984, 990, 986,
    # Zuid Holland
    956, 953, 945, 958, 954, 951, 959, 955, 952, 960, 957, 949, 950
]

DISABLE_CNNS_ID = [connection_id_gen(t=M2M, x=idx) for idx in DISABLE_CNNS_IDX]

# Track unused connections before dropping
unused_cnns_df = intrapv_cnns_df.loc[DISABLE_CNNS_ID].copy()
intrapv_cnns_df.drop(DISABLE_CNNS_ID, inplace=True)

for province in netherlands.provinces:
    if DEBUG_MODE:
        build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[2000].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=2000,
            connections_df=pd.concat(
                [intrapv_cnns_df,sources_cnns_df],
                axis=0,
                join='outer'
            ),
            dates_as_str=False,
            pipe_options_df=pipe_options,
            pipes_roughnesses_history=None,
            situation_name='checkpoint_3'
        ).close()

### Remove excess connections 

We use an heuristic to remove excess connections based on the municipality size (population).


In [ ]:
for province in netherlands.provinces:
    maks_pv = intrapv_cnns_df['province'] == province.cbs_id

    intrapv_cnns_df.loc[maks_pv] = define_cnns.topology_heuristic(
        df=intrapv_cnns_df.loc[maks_pv],
        a_province=province
    )

    if DEBUG_MODE:
        build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[2000].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=2000,
            connections_df=pd.concat(
                [intrapv_cnns_df,sources_cnns_df],
                axis=0,
                join='outer'
            ),
            dates_as_str=False,
            pipe_options_df=pipe_options,
            pipes_roughnesses_history=None,
            situation_name='checkpoint_4',
            pda=True,
            future_sources=True
        ).close()

### Manual fixes after heuristic 

We look at the networks again and decide if we need to re-activate or de-activate connections at this point

In [ ]:
# Manual fixes
DEACTIV_CNNS_IDXS = [
    # Drenthe
    # Flevoland
    261,
    # Fryslan
    94, 132, 128, 97, 102, 112, 121,
    # Gelderland (divide by the river)
    401, 290, 331, 314, 425, 396,
    # Groningen
    # Limburg
    1260, 1269,
    # Noord-brabant
    # Noord-holland
    # Overijssel
    # Utrecht
    # Zeeland
    991, 985
    # Zuid-holland
]
REACTIV_CNNS_IDXS = [
    # Drenthe
    # Flevoland
    # Fryslan
    # Gelderland
    # Groningen
    # Limburg
    # Noord-brabant
    # Noord-holland
    686,
    667, # This is necessary to allow the flow move around Amsterdam
    # Overijssel
    # Utrecht
    # Zeeland
    # Zuid-holland
]

DEACTIV_CNNS_ID = [connection_id_gen(t=M2M, x=idx) for idx in DEACTIV_CNNS_IDXS]
REACTIV_CNNS_ID = [connection_id_gen(t=M2M, x=idx) for idx in REACTIV_CNNS_IDXS]
intrapv_cnns_df.loc[intrapv_cnns_df.index.isin(DEACTIV_CNNS_ID), ['pipes-installation_dates', 'pipes-option_ids']] = [pd.NaT, ""]

reactivate_mask = intrapv_cnns_df.index.isin(REACTIV_CNNS_ID)
intrapv_cnns_df.loc[reactivate_mask, 'pipes-installation_dates'] = intrapv_cnns_df.loc[reactivate_mask, 'min_date']
intrapv_cnns_df.loc[reactivate_mask, 'pipes-option_ids'] = UNDECIDED_PIPE_OPTION

for province in netherlands.provinces:
    if DEBUG_MODE:
        build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[2000].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=2000,
            connections_df=pd.concat(
                [intrapv_cnns_df,sources_cnns_df],
                axis=0,
                join='outer'
            ),
            dates_as_str=False,
            pipe_options_df=pipe_options,
            pipes_roughnesses_history=None,
            situation_name='checkpoint_5',
            pda=True
        ).close()

# We have decided the topology. Drop unused connections and keep only the one used up to this point.
# If we need one of these connections, we will re-define them with the 2025 topology
TOPOLOGY_200_MASK = intrapv_cnns_df['pipes-option_ids'] != ''
unused_cnns_df = pd.concat([
    unused_cnns_df,
    intrapv_cnns_df.loc[~TOPOLOGY_200_MASK]
])
intrapv_cnns_df = intrapv_cnns_df[TOPOLOGY_200_MASK]

### Adjust the ages

We change the installation dates to meet the desired average age in 2000.

In [ ]:
# A few of the a-posteriori analysis is necessary 
# Adjust the installation dates so that average age is approx 50 (1% replacemnt rate)
all_cnns_df = pd.concat([intrapv_cnns_df, sources_cnns_df], axis=0, join='outer')
# Let's leave pipes that have as min date after 1950 (would have 75 years in 2025) as they are.
# We get how many thery are so that we can spread the others (before 1950) to match the average age that we want
TYEAR=pd.to_datetime("1950-01-01")
Y2000=pd.to_datetime("2000-01-01")
mask_installed = all_cnns_df['pipes-option_ids'].notna() & (all_cnns_df['pipes-option_ids'] != '')
mask_postT = (all_cnns_df['min_date'] >= TYEAR)
mask_range = all_cnns_df['min_date'].between(TYEAR, Y2000)

age_ok = Y2000 - all_cnns_df.loc[mask_installed & mask_range, 'pipes-installation_dates']
avg_age_ok = age_ok.mean() / pd.Timedelta(days=365.25)
n_age_ok = len(age_ok)
print(f"{n_age_ok} pipes are installed between 1950 and 2025 with average age in 2000 of {avg_age_ok}")

age_toadjust = Y2000 - all_cnns_df.loc[mask_installed & ~mask_postT, 'pipes-installation_dates']
avg_age_toadjust = age_toadjust.mean() / pd.Timedelta(days=365.25)
n_age_toadjust = len(age_toadjust)
print(f"While there are {n_age_toadjust} connections to adjust the ages (right now their average age in 2000 is {avg_age_toadjust}).")

# if we want replacement rate is 1% every 1% of connection has age: 0, 1, 2, and so on. thus the average is 49,5
TARGET_AGE = 49.5 
TARGET_AGE_TOADJUST = (TARGET_AGE * (n_age_ok+n_age_toadjust) - avg_age_ok * n_age_ok) / n_age_toadjust
print(f"so the old pipes should have an average age of {TARGET_AGE_TOADJUST}")
UB_TAGE = 80 # We want at most 80 years old pipes 
LB_TAGE = TARGET_AGE_TOADJUST-(UB_TAGE-TARGET_AGE_TOADJUST)
RNG_AGES = RNG.spawn(n_children=1)[0]
new_ages = RNG_AGES.integers(low=LB_TAGE, high=UB_TAGE, size=n_age_toadjust)
all_cnns_df.loc[mask_installed & ~mask_postT, 'pipes-installation_dates'] = [pd.Timestamp(year=2000-age, month=1, day=1) for age in new_ages]

invalid_date_mask = (
    mask_installed & 
    (all_cnns_df['pipes-installation_dates'] < all_cnns_df['min_date'])
)
all_cnns_df.loc[invalid_date_mask, 'pipes-installation_dates'] = all_cnns_df.loc[invalid_date_mask, 'min_date']

### Decide the pipes darcy friction factor in 2000

This part of the netboook used to be separated in a "pipes-dynamic_properties", after we decided the pipe options.

However, at this point in the process, we know the age of the connection's pipe in 2000, but we don't know all the pipe's option. 
Therefore, we can't know the friction factor of every pipe in 2000, because it depends on the type.

What we can do, however, is prepare the relative decay rates in the history of the pipe, every year a value between 0 (min decay) and 1 (max decay).
Then, once the option is chosen (this means also inside the optimisation) we have:

pipe friction factor in 2000 = 
- pipe option "new pipe friction factor" + 
- pipe option "minimum decay rate" * *age of the pipe in 2000* + 
- *cumulative relative decay* * pipe option (maximum decay factor - minimum decay factor)

The optimisaton will need to change the roughness as well as the diameter, becasue that one parameter (pipe option)
influences both variables.

For the ones that we know (e.g. source) calculate it already and put it in a dataframe to track the history.

In [ ]:
RNG_DFF = default_rng(12822)

Y2000 = pd.Timestamp(year=2000, month=1, day=1)

# Ordered dict of pipes, pipes have id: '{connection_id}-{idx}' where idx is created with enumerate starting from 0
# At this point, we only have one pipe per connection and a possible installation date
pipes_rghnss = {}
cnns_rel_decay = {}
cnns_age_in_2000 = {}
for connection_id, connection_data in all_cnns_df.iterrows():
    pipe_install_date = connection_data['pipes-installation_dates']

    if pd.isna(pipe_install_date) or pipe_install_date > Y2000:
       continue

    pipe_option_id = connection_data['pipes-option_ids']
    pipe_age = int((Y2000 - pipe_install_date).days / 365.25)

    if pipe_option_id == UNDECIDED_PIPE_OPTION:
        rel_decays = RNG_DFF.spawn(1)[0].uniform(low=0, high=1, size=pipe_age).astype(float)

        cnns_rel_decay[connection_id] = sum(rel_decays)
        cnns_age_in_2000[connection_id] = pipe_age

    else:
        pipe_id = f"{connection_id}-00"
            
        pipe_rgh = pipe_options.loc[str(pipe_option_id), 'darcy_friction_factor-new_pipe'].astype(float)
        pipe_rgh_decay_bounds = (
            pipe_options.loc[str(pipe_option_id), 'darcy_friction_factor-decay_rate-min'].astype(float),
            pipe_options.loc[str(pipe_option_id), 'darcy_friction_factor-decay_rate-max'].astype(float)
        )

        pipe_rgh_decays = RNG_DFF.spawn(1)[0].uniform(*pipe_rgh_decay_bounds, size=pipe_age)
        
        pipes_rghnss[pipe_id] = round(pipe_rgh + pipe_rgh_decays.sum(), 6)

pipes_rgh_df = pd.DataFrame.from_dict({Y2000: pipes_rghnss}, orient='index', dtype=float)
pipes_rgh_df.index.name = 'timestamp'

display(pipes_rgh_df)

## Optimise the pipe diameters

We run an optimisation for each water utility where we optimise each water utility selected pipes.


In [ ]:
import opt_utils
from multiprocessing import Pool
import opt_state
opt_results: dict[Province, opt_utils.OptimisationResult] = {}
opt_metrics: dict[Province, opt_utils.OptimisationMetrics] = {}

RUN_OPTIMISATION = True
N_ISLANDS = 10
NO_PIPE_OPTION = False

opt_args = {}
for province in sorted(netherlands.provinces, key=lambda pv: pv.cbs_id):
    # if not province.matches('drenthe'):
    #      continue
    pv_seed = RNG.spawn(n_children=1)[0]
    opt_seeds = pv_seed.spawn(n_children=2) # one for algo and one for population
    ALGO_SEEDS = opt_seeds[0].integers(0, 2**32 - 1, size=N_ISLANDS)
    POP_SEEDS = opt_seeds[1].integers(0, 2**32 - 1, size=N_ISLANDS)
    for isl in range(N_ISLANDS):
        opt_args[(province, isl)] = (
            RUN_OPTIMISATION,
            province.outfiles_name,
            province.cbs_id,
            isl,
            int(ALGO_SEEDS[isl]),
            int(POP_SEEDS[isl])
        )

with Pool(
    initializer=opt_state.init_worker,
    initargs=(
        settings_desc,
        nl_desc,
        sources_desc,
        pumping_desc,
        juris_max_hourly_dem_df.loc[2000].to_dict(),
        all_cnns_df.copy(),
        pipe_options,
        pipes_rgh_df,
        cnns_rel_decay,
        cnns_age_in_2000,
        NO_PIPE_OPTION
    )
) as pool:
    pool_results = pool.starmap(
        opt_utils.optimise_province_cnns,
        opt_args.values(),
        chunksize=1
    )

for province, pool_result in zip(
    [key[0] for key in opt_args],
    pool_results
):
    if province not in opt_results:
        opt_results[province] = pool_result
    else:
        opt_results[province].decisions.update(pool_result.decisions)
        opt_results[province].fitnesses.update(pool_result.fitnesses)

In [ ]:
for i, province in enumerate(list(netherlands.provinces)):
    if opt_results[province]:
        opt_metrics[province] = opt_utils.compute_final_metrics(
            opt_results[province]
        )

import plot_utils
plot_utils.plot_hypervolumes(netherlands, opt_metrics).show()
plot_utils.plot_final_paretos(netherlands, opt_metrics).show()

In [ ]:
FORCE_BACKUP = False
backup_df = pd.read_excel('../.backups/connections-static_properties-backup-2025-11-23.xlsx', sheet_name='provincial', index_col='connection_id')
backup_df['pipes-option_ids'] = backup_df['pipes-option_ids'].astype(str).apply(lambda x: '' if x=='nan' else x)

SELECT_INDEXES_MAPS = {
    "Groningen": 3,
    "Drenthe": 1,
    "Fryslan": 2,
    "Flevoland": 25,
    "Overijssel": 4,
    "Gelderland": 0,
    "Utrecht": 5,
    "Noord-brabant": 16,
    "noord-holland": 0,
    "zuid-holland": 1,
    "zeeland": 6,
    "limburg": 11,
}

pumping_station_speeds = {}

for pv_name, idx in SELECT_INDEXES_MAPS.items():
    province = netherlands.province(pv_name)
    pumping_station_speeds[province] = {}
    
    # If we don't have a fresh optimisation, use the backup
    if FORCE_BACKUP or province not in opt_results or province not in opt_metrics:
        opt_utils.replace_options_with_backup(
            all_connections_df=all_cnns_df,
            backup_df=backup_df,
            a_province=province,
            pipes_roughnesses_history=pipes_rgh_df,
            cnns_rel_decay_history=cnns_rel_decay,
            cnns_ages_in_2000=cnns_age_in_2000
        )
    else:
        pumping_station_speeds[province] = opt_utils.replace_options_with_selected_solution(
            all_connections_df=all_cnns_df,
            a_province=province,
            opt_results=opt_results[province],
            opt_metrics=opt_metrics[province],
            select_sol_idx=idx,
            pipes_roughnesses_history=pipes_rgh_df,
            cnns_rel_decay_history=cnns_rel_decay,
            cnns_ages_in_2000=cnns_age_in_2000
        )

    MASK_PV_CNNS = all_cnns_df['province']==province.cbs_id
    if DEBUG_MODE:
        build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[2000].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=2000,
            connections_df=all_cnns_df[MASK_PV_CNNS],
            dates_as_str=False,
            pipe_options_df=pipe_options,
            pipes_roughnesses_history=pipes_rgh_df,
            situation_name='checkpoint_6',
            pda=True,
            future_sources=False,
            pumping_station_representation="free_parallel_pumps",
            pumping_stations_speeds=pumping_station_speeds[province]
        ).close()


In [ ]:
from water_futures_battle.core.base_model import DynamicProperties
from water_futures_battle.pipes.dynamic_properties import PipesDB

pipes_dps = DynamicProperties(PipesDB.NAME)

pipes_dps[PipesDB.FRICTIONF] = pipes_rgh_df.copy()

pipes_dps.dump('../../pipes')

## Complete system definition

New municipalities appear, so we define new connections for them that replace old connections.



In [ ]:
# Check if we need to define new connections for new municipalities that open in the future
n_nl_new_intrapv_cnns = 1 # National counter for connections that appear after 2000
shift_new_intrapv_cnns = 2000 # These new connections will have an id starting with 2000
new_intrapv_cnns_options = {  # force for some of the new connections a given pipe option
    "CG2005": 'PI02',
    "CG2006": 'PI03',
    "CG2007": 'PI03',
    "CG2015": None, # Removing this pipe is crucial to have a robust Limburg system
    "CG2018": 'PI02',
    "CG2019": 'PI02',
    "CG2025": 'PI04',
    "CG2034": 'PI06',
    "CG2041": 'PI01',
    "CG2045": 'PI03',
    "CG2049": 'PI03',
    "CG2055": 'PI03',
    "CG2059": 'PI01',
    "CG2067": 'PI05',
    "CG2075": 'PI03',
    "CG2082": 'PI05',
    "CG2098": 'PI02',
    "CG2100": 'PI02',
    "CG2107": 'PI03',
    "CG2127": 'PI08',
    "CG2130": 'PI08',
    "CG2134": 'PI09',
    "CG2141": 'PI04',
    "CG2146": 'PI03',
    "CG2151": 'PI02',
    "CG2153": 'PI01',
    "CG2164": 'PI03',
    "CG2165": 'PI09',
}
for year in list(YEARS):
    for province in sorted(netherlands.provinces, key=lambda pv: pv.cbs_id):

        # Variable to decide if we need to save the network (only if new municipality appears)
        situation_name = None
        if year != list(YEARS)[0]:
            temp_df = define_cnns.appearing_municipalities(
                a_province=province,
                year=year,
                all_existing_connections_df=all_cnns_df,
                peak_demands_df=juris_max_hourly_dem_df
            )
            temp_df['connection_id'] = [
                connection_id_gen(t=M2M, x=shift_new_intrapv_cnns +i)
                for i in range(
                    n_nl_new_intrapv_cnns,
                    n_nl_new_intrapv_cnns+len(temp_df)
                )
            ]

            if not temp_df.empty:
                # Some new connections appeared, it means that it replaced some old one
                # We need to update the "replaced_by" value of the replaced connection
                # as the function appearing_municipalities leaves a "SOMEONE" placeholder
                # Also, we implement a bit of logic for these replacements:
                #     1. the connection replaces more than 1 connection
                #        -> mandatory new pipe to be installed manually deciding the option
                #     2. the connection replaces a connection that has a pipe with age > its min lifetime
                #        -> replace
                temp_df['province'] = province.cbs_id
                temp_df.set_index('connection_id', inplace=True)
                for replacing_cnn_id, replacing_cnn in temp_df.iterrows():
                    replaced_cnns = replacing_cnn['replaces'].split(';')

                    all_cnns_df.loc[
                        replaced_cnns,
                        'replaced_by'
                    ] = str(replacing_cnn_id)

                    # It could be that a replaced cnns is also a replacement of a previous
                    # connection but it has not replaced the pipe yet, in that case,
                    # we replace the old pipe.
                    effective_replaced_cnns = []
                    for c in replaced_cnns:
                        if all_cnns_df.loc[c, 'pipes-option_ids'] != '':
                            # base case: replaces a connection with a pipe, that is the effective one
                            effective_replaced_cnns.append(c)
                            continue
                    
                        # if we are here it means we are replacing a connection without a pipe
                        # it should be a replacement connection (ID > 2000).
                        # However, this replacement connection should replace only 
                        # one other pipe because connections replacing more than one
                        # get a pipe option immediately.
                        re_replaced_cnns = all_cnns_df.loc[c, 'replaces'].split(';')
                        assert len(re_replaced_cnns) == 1, "A re-replacement should only be replacing a single connection"
                        
                        if pd.isna(all_cnns_df.loc[re_replaced_cnns[0], 'last_pipe_decom_date']):
                            # if is the old one that is still active, let's add it to the list
                            effective_replaced_cnns.append(re_replaced_cnns[0])
                        else:
                            # otherwise we couldn't find a re-replacement
                            effective_replaced_cnns.append(c)

                    if len(replaced_cnns) > 1:
                        # We are in case 1 of the replacement logic

                        if replacing_cnn_id not in new_intrapv_cnns_options:
                            new_intrapv_cnns_options[replacing_cnn_id] = UNDECIDED_PIPE_OPTION    

                        if new_intrapv_cnns_options[replacing_cnn_id] == UNDECIDED_PIPE_OPTION:
                             # Notify me if I haven't dealt with it already
                            print(f"We need to decide the diameter of connection {replacing_cnn_id} in province {province.display_name} and year {year}")
                            print(f"  This connection replaces:")
                            for c in replaced_cnns:
                                print(f"   - Cnn: {c}, with diameter {all_cnns_df.loc[c, 'pipes-option_ids']}")

                        # close the replaced connections on the replacing connection's year
                        # and set the new connection option from the manually chosen value

                        all_cnns_df.loc[
                            effective_replaced_cnns,
                            'last_pipe_decom_date'
                        ] = replacing_cnn['min_date']

                        if new_intrapv_cnns_options[replacing_cnn_id] is None:
                            # it means we don't want to activate this new connection
                            temp_df.loc[
                                replacing_cnn_id,
                                ['pipes-installation_dates', 'pipes-option_ids']
                            ] = [
                                pd.NaT,
                                ''
                            ]
                        else:
                            # if we are here it means we want to active it
                            temp_df.loc[
                                replacing_cnn_id,
                                ['pipes-installation_dates', 'pipes-option_ids']
                            ] = [
                                replacing_cnn['min_date'],
                                new_intrapv_cnns_options[replacing_cnn_id]
                            ]
                        
                        # done, move to next replacing connection
                        continue

                    # if we are here, this replacing connection replaces only one
                    replaced_cnn_id = effective_replaced_cnns[0]
                    replaced_pipe_age = (year-all_cnns_df.loc[replaced_cnn_id, 'pipes-installation_dates'].year)
                    replaced_pipe_option = all_cnns_df.loc[replaced_cnn_id, 'pipes-option_ids']

                    if replaced_pipe_option == '':
                        # e.g. replacing a new connection which has not been used yet
                        print(f"Connection {replacing_cnn_id} (Province {province.display_name}) replaces {replaced_cnns[0]}, which is empty",
                              f"  because it replaces {all_cnns_df.loc[replaced_cnns[0], 'replaces']}")
                        continue

                    if replaced_pipe_option == UNDECIDED_PIPE_OPTION:
                        # e.g. replacing a connection that is replacing another one
                        print(f"Connection {replacing_cnn_id} replaces (Province {province.display_name}) {replaced_cnns[0]}, which is still undecided.")
                        continue

                    # If the current age of the pipe in the connection being replaced
                    # is above its minimum lifetime, we can assume that the water 
                    # utility would have replaced it automatically
                    min_lifetime = pipe_options.loc[replaced_pipe_option, 'lifetime-min']

                    if replaced_pipe_age >= min_lifetime:
                        replacing_cnn_date = pd.Timestamp(year=year, month=1, day=1)

                        all_cnns_df.loc[
                            replaced_cnn_id,
                            'last_pipe_decom_date'
                        ] = replacing_cnn_date

                        # See if we can use the replaced pipe option or we are overwriting it
                        if replacing_cnn_id in new_intrapv_cnns_options:
                            replacing_cnn_option = new_intrapv_cnns_options[replacing_cnn_id]

                            if replacing_cnn_option is None:
                                replacing_cnn_option = ''
                                replacing_cnn_date = pd.NaT
                        else:
                            replacing_cnn_option = replaced_pipe_option

                        temp_df.loc[
                            replacing_cnn_id,
                            ['pipes-installation_dates', 'pipes-option_ids']
                        ] = [
                            replacing_cnn_date,
                            replacing_cnn_option
                        ]
                    
                all_cnns_df = pd.concat([all_cnns_df, temp_df], axis=0)

                n_nl_new_intrapv_cnns += len(temp_df)

                # Save the network too (if in debug)
                situation_name = 'checkpoint_7' if DEBUG_MODE else None

        MASK_PV_CNNS = all_cnns_df['province']==province.cbs_id
        build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[year].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=year,
            connections_df=all_cnns_df[MASK_PV_CNNS],
            dates_as_str=False,
            pipe_options_df=pipe_options,
            pipes_roughnesses_history=pipes_rgh_df,
            situation_name=situation_name,
            pda=True,
            future_sources=False,
            pumping_station_representation="free_parallel_pumps",
            pumping_stations_speeds=pumping_station_speeds[province]
        ).close()

In [ ]:
# Finally simulate the network at peak demands to see if we need any manual adjustment to do before, the end.
def sample_end_dates(
        row: pd.Series,
        pipe_options_df: pd.DataFrame,
        lookup_df: pd.DataFrame
    ) -> pd.Timestamp:
    
    pipe_option_id = row['pipes-option_ids']

    # Placeholder for empty connections or connections that will close anyway.
    if (
        pipe_option_id == '' or
        pipe_option_id == UNDECIDED_PIPE_OPTION or
        pd.notna(row['last_pipe_decom_date'])
    ):
        return pd.Timestamp(year=2100, month=1, day=1)

    sampled_lifetime = RNG.integers(
        int(pipe_options_df.loc[pipe_option_id, 'lifetime-min']),
        int(pipe_options_df.loc[pipe_option_id, 'lifetime-max'])
    )

    sampled_end_date = row['pipes-installation_dates'] + pd.DateOffset(years=int(sampled_lifetime))

    # Adjust in case we got a short lifetime
    while sampled_end_date <= pd.Timestamp(year=2000, month=1, day=1):
        sampled_end_date += pd.DateOffset(years=5)

    # We have sampled the end date, however, we want to avoid replacing a pipe just a few
    # years before we replace the connection.
    replacing_cnn_id = str(row['replaced_by'])

    if replacing_cnn_id == '':
        # connection doesn't get replaced, the sampled end date is legit
        return sampled_end_date 
    
    if lookup_df.loc[replacing_cnn_id, 'replaced_by'] != '':
        # replacement connection is also being replaced...
        # make the re-replaced connection the replacing one (unless the re-replacing one is a double)
        re_replacing_cnn_id = lookup_df.loc[replacing_cnn_id, 'replaced_by']
        
        if re_replacing_cnn_id == 'SELF_LOOP':
            return sampled_end_date

        if len(lookup_df.loc[re_replacing_cnn_id, 'replaces'].split(';')) == 1:
            print(f"Shifted replacing connection of connection {row.name}, i.e., connection {replacing_cnn_id}")
            print(f" to {re_replacing_cnn_id}")

        replacing_cnn_id = re_replacing_cnn_id
    
    # the connection get's replaced, if before the new connection appears (i.e.,
    # min date of replacing), we "hold on" until and delay it
    if sampled_end_date < lookup_df.loc[replacing_cnn_id, 'min_date']:
        print(f"requested connection {row.name} to go off on {sampled_end_date}")
        print(f"at {sampled_end_date.year-pd.to_datetime(row['pipes-installation_dates']).year} age.")
        print(f"Delayed until {lookup_df.loc[replacing_cnn_id, 'min_date']}")

        return lookup_df.loc[replacing_cnn_id, 'min_date']
    
    # else, is fine use the sampled end date
    return sampled_end_date

sampled_end_dates = all_cnns_df.apply(
    lambda row: sample_end_dates(
        row=row,
        pipe_options_df=pipe_options,
        lookup_df=all_cnns_df.copy()
    ),
    axis=1
)
assert all(sampled_end_dates > pd.Timestamp(year=2000, month=1, day=1))

all_cnns_df['pipes-installation_dates'] = all_cnns_df['pipes-installation_dates'].apply(
    lambda x: x.strftime('%Y-%m-%d') if pd.notna(x) else ''
)

from epanet_utils import get_total_undelivered_demand
undelivered_demands: dict[tuple[str, int], float] = {} # Track undelivered demands across years and utilities
for year in YEARS:
    for province in netherlands.provinces: 

        MASK_PV_CNNS = all_cnns_df['province'] == province.cbs_id

        # Modify the connections df adding the new pipes and 
        pipes_rgh_df, all_cnns_df = define_cnns.install_new_pipes(
            connections=all_cnns_df,
            pipes_rgh_history=pipes_rgh_df,
            pipes_options=pipe_options,
            year=year,
            mask_pv_cnns=MASK_PV_CNNS,
            sampled_end_dates=sampled_end_dates
        )

        temp_net = build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[year].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=year,
            connections_df=all_cnns_df[MASK_PV_CNNS],
            dates_as_str=True,
            pipe_options_df=pipe_options,
            pipes_roughnesses_history=pipes_rgh_df,
            situation_name='checkpoint_8' if DEBUG_MODE else None,
            pda=True,
            future_sources=False,
            pumping_station_representation="free_parallel_pumps",
            pumping_stations_speeds=pumping_station_speeds[province],
            epyt_kwargs={
                'ignore_error_codes': [4, 5, 6]
            }
        )

        ud = get_total_undelivered_demand(temp_net)
        undelivered_demands[(province.cbs_id, year)] = max(ud, 0.0)

        if DEBUG_MODE and (province.cbs_id, year-1) in undelivered_demands:
            if undelivered_demands[(province.cbs_id, year-1)] < undelivered_demands[(province.cbs_id, year)]:
                amount = undelivered_demands[(province.cbs_id, year)] - undelivered_demands[(province.cbs_id, year-1)]
                if undelivered_demands[(province.cbs_id, year-1)] == 0.0:
                    percentamount = float('inf')
                else:
                    percentamount = 100 * amount / undelivered_demands[(province.cbs_id, year-1)]
                print(f"UD worsening ({year} | {province.cbs_id}) by {amount:.2f} ({percentamount:.2f}%)")

        temp_net.close()

        # Increase the friction factor for next year
        pipes_rgh_df = define_cnns.age_pipes(
            connections=all_cnns_df[MASK_PV_CNNS],
            pipes_rgh_history=pipes_rgh_df,
            pipes_options=pipe_options,
            year=year,
            pipe_ageing_rng=settings.pipes_frict_f_decay_rng
        )

plot_utils.plot_undelivered_demands(undelivered_demands).show()

In [ ]:
# Manual adjustemtns 
# CG2013 replaces CG1206 but becomes a self loop. We didn't deactivate the second one, and the first one is never activated
all_cnns_df.loc["CG1206", "last_pipe_decom_date"] = all_cnns_df.loc["CG2013", "last_pipe_decom_date"]
all_cnns_df.loc["CG2013", "last_pipe_decom_date"] = pd.NaT

# Cross-provincial connections

We define the inter-province (cross-provincial) connections manually

In [ ]:
new_connections = [
    # Groningen, Drenthe
    ('GM0014', 'GM1730'),
    ('GM0037', 'GM1681'),
    # Fryslan, Groningen
    ('GM0059', 'GM1969'),
    # Drenthe, Fryslan
    ('GM1699', 'GM0085'),
    ('GM0098', 'GM1701'),
    # Overijssel, Drenthe
    ('GM1896', 'GM0119'),
    ('GM0118', 'GM0160'),
    # Zuid holland, Zeeland
    ('GM1924', 'GM1676'),
    ('GM1924', 'GM0716'),
    # Utrecht, Zuid holland
    ('GM0589', 'GM1931'),
    ('GM0632', 'GM1901'),
    # Zuid holland, Noord holland
    ('GM0534', 'GM0397'),
    # Noord holland, Utrecht
    ('GM0402', 'GM0308'),
    ('GM0451', 'GM0736'),
    # Noord holland, Flevoland
    ('GM1942', 'GM0034'),
    # Flevoland Gelderland,
    ('GM0050', 'GM0243'),
    # Flevoland, Overijssel
    ('GM0303', 'GM0166'),
    # Flevoland, Fryslan
    ('GM0171', 'GM1940'),
    # Zeeland, Noord Brabant
    ('GM0716', 'GM0748'),
    ('GM0716', 'GM0851'),
    # Overijssel, Gelderland
    ('GM0150', 'GM0285'),
    ('GM0193', 'GM0244'),
    ('GM0158', 'GM1859'),
    # Noord Brabant, Zuid Holland
    ('GM1959', 'GM0512'),
    # Gelderland, Limburg
    ('GM0252', 'GM0944'),
    # Limburg, Noord brabant
    ('GM0944', 'GM1982'),
    ('GM0988', 'GM1706'),
    # Gelderland, Utrecht
    ('GM0279', 'GM0351'),
    ('GM0216', 'GM1961'),
    ('GM0289', 'GM0340'),
    # Gelderland, Noord brabant
    ('GM0263', 'GM0796'),
    ('GM0668', 'GM0828')
]

cross_connections_df = pd.DataFrame(
    data=new_connections,
    columns=['from_node', 'to_node']
)
cross_connections_df['connection_id'] = [connection_id_gen('PROVINCE', i) for i in range(1, len(new_connections)+1)]
cross_connections_df['distance'] = cross_connections_df.apply(
    lambda row: round(
        define_cnns.compute_distance(
            netherlands.municipality(row['from_node']),
            netherlands.municipality(row['to_node']),
        ),
        ndigits=2
    ),
    axis=1
)
cross_connections_df['minor_loss_coefficient'] = 0
cross_connections_df['pipes-installation_dates'] = pd.NaT
cross_connections_df['pipes-option_ids'] = ""
cross_connections_df['pipes-decommission_dates'] = pd.NaT
cross_connections_df['replaced_by'] = ""
cross_connections_df['replaces'] = ""
cross_connections_df['usable_in_2025'] = True

# Put the connection id as a first column
cross_connections_df.set_index('connection_id', inplace=True)
cross_connections_df.reset_index(inplace=True)

## New connections in 2025

Redefine a full delaunay triangulation in 2025, to see what are the existing connections.

In [ ]:
n_nl_2025_new_intrapv_cnns = 1 # National counter for connections that we define in 2025
shift_2025_intrapv_cnns = 3000 # These new connections will have an id starting with 3000

# After a first run we know we are going to remove the following ones:
DISABLE_2025_CNNS_IDXS = [
    # Drenthe
    # Flevoland
    # Frieslan
    56, 53, # create crossings
    # Gelderland
    304, 303, 301, 297, 290,
    295, 289, 254, # crossings
    # Groningen
    21, 20, 19,
    # Limburg
    884, 872, 882, 879,
    867, #crossing
    # N-brabant
    809, 806, 805, 807,
    777, # crossing
    # N-Holland
    482, 473, 460,
    469, # crossing
    # Overijssel
    153, 149,
    140, # crossing
    # Utrecht
    370, 367, 354, 351, 365,
    359, # crossing
    # Zeeland
    657, 
    # Zuid Holland
    627, 624, 616, 628, 625, 620, 626, 623,
    617, 613, 569, 610, 612, 598, 582, 604, # crossings
]
DISABLE_2025_CNNS_IDS = [
    connection_id_gen(t=M2M, x=idx + shift_2025_intrapv_cnns)
    for idx in DISABLE_2025_CNNS_IDXS
]

def get_corresponding_cnn_id(
        cnn: pd.Series,
        lookup_df: pd.DataFrame
) -> str:
    """
    Checks if a connection with the same from node-to node is already defined 
    in the lookup_df.
    Return self if not found
    """
    from_node = cnn['from_node']
    to_node = cnn['to_node']

    def get_lookup_nodes(row: pd.Series) -> set[str]:
        lookup_nodes = {str(row['from_node']), str(row['to_node'])}
    
        for i in range(2):
            extra_nodes = set()
            for n in lookup_nodes:
                if not n.startswith('GM'):
                    continue
                m = netherlands.municipality(n)
                if m.end_reason == 'renamed':
                    extra_nodes.add(m.destination_municipality_cbs_id)
            lookup_nodes.update(extra_nodes)
        
        return lookup_nodes
        
    match = lookup_df[
        lookup_df.apply(lambda row: 
            {from_node, to_node}.issubset(get_lookup_nodes(row)), axis=1)
    ]
    
    if not match.empty:
        return match.index[0]
    
    return cnn['connection_id']

for province in sorted(netherlands.provinces, key=lambda pv: pv.cbs_id):
    temp_df = define_cnns.full_delaunay(
        a_province=province,
        handled_municipalities=handled,
        year=2025,
        default_option_id=UNDECIDED_PIPE_OPTION
    )
    temp_df['connection_id'] = [
        connection_id_gen(t=M2M, x=i+shift_2025_intrapv_cnns)
        for i in range(
            n_nl_2025_new_intrapv_cnns,
            n_nl_2025_new_intrapv_cnns+len(temp_df)
        )
    ]
    n_nl_2025_new_intrapv_cnns += len(temp_df)

    # First of all, these connections will not be used
    # we put strings and not nat because the rest of the times are already as str 
    temp_df['pipes-installation_dates'] = "2025-01-01"
    temp_df['pipes-option_ids'] = UNDECIDED_PIPE_OPTION

    # Check that the connections are not existing already, in case drop them
    MASK_EXISTING = temp_df.apply(
        lambda row: get_corresponding_cnn_id(row, all_cnns_df) != row['connection_id'],
        axis=1
    )
    temp_df = temp_df[~MASK_EXISTING]

    # Check that the connection was not defined already, but simply unused
    # in that case, use the old unused connection name and drop it there 
    temp_df['connection_id'] = temp_df.apply(
        lambda row: get_corresponding_cnn_id(row, unused_cnns_df),
        axis=1
    )

    # For sure, if we decided to remove this pipe before completely because it is
    # a connection that doesn't make sense, we throw it away again
    MASK_OLD_REMOVED = temp_df['connection_id'].isin(DISABLE_CNNS_ID)
    temp_df = temp_df[~MASK_OLD_REMOVED]

    MASK_NEW_REMOVED = temp_df['connection_id'].isin(DISABLE_2025_CNNS_IDS)
    temp_df = temp_df[~MASK_NEW_REMOVED]

    temp_df = temp_df.set_index('connection_id')
    temp_df['province'] = province.cbs_id

    if not temp_df.empty:
        all_cnns_df = pd.concat([all_cnns_df, temp_df], axis=0)

    if DEBUG_MODE:
        MASK_PV_CNNS = all_cnns_df['province'] == province.cbs_id
        build_epyt(
            province=province,
            munic_demands=juris_max_hourly_dem_df.loc[2024].to_dict(),
            all_sources=all_sources,
            pump_options=pump_options,
            all_pumping_stations=pumping_stations,
            year=2025,
            connections_df=all_cnns_df[MASK_PV_CNNS],
            dates_as_str=True,
            pipe_options_df=pipe_options,
            pipes_roughnesses_history=pipes_rgh_df,
            situation_name='checkpoint_9',
            pda=True,
            future_sources=True,
            pumping_station_representation="free_parallel_pumps",
        ).close()

# After we have gejerated the checkpoint netwokrs we still have UNDECIDED in the new connections
MASK_UNDECIDED = all_cnns_df['pipes-option_ids'] == UNDECIDED_PIPE_OPTION
all_cnns_df.loc[MASK_UNDECIDED, ['pipes-installation_dates', 'pipes-option_ids']] = ['', '']

In [ ]:
# Save out 
from water_futures_battle.core.base_model import StaticProperties
connections_sp = StaticProperties('connections-static_properties')

from water_futures_battle.connections.entities import Connection

df = all_cnns_df.reset_index()
df = df.drop(columns=define_cnns.COLUMNS_CONNECTION[-4:]+['province'])
df['distance'] = df['distance'].round(2)
def build_decommission_dates(row):
    install_str = row['pipes-installation_dates']
    last_date = row['last_pipe_decom_date']
    
    # Parse the installation dates
    if pd.isna(install_str) or install_str == '':
        dates = []
    else:
        dates = [d.strip() for d in install_str.split(';') if d.strip()]
    
    # Shift by one: drop the first, keep the rest
    shifted = dates[1:] if len(dates) > 1 else []
    
    # Append the last pipe decommission date
    if pd.notna(last_date):
        shifted.append(last_date.strftime('%Y-%m-%d'))
    
    return ';'.join(shifted)
df['pipes-decommission_dates'] = df[['pipes-installation_dates' ,'last_pipe_decom_date']].apply(
    build_decommission_dates,
    axis = 1
)
df = df[[Connection.ID, Connection.FROM_NODE, Connection.TO_NODE, Connection.DISTANCE,
         Connection.MINORLOSSC, Connection.P_OPTION, Connection.P_INSTDATE, Connection.P_DECODATE,
         Connection.REPLACED_BY, Connection.REPLACES, 'usable_in_2025']]
provincial_mask = df['connection_id'].str.startswith('CG')

connections_sp['provincial'] = df[provincial_mask]
connections_sp['sources'] = df[~provincial_mask]
connections_sp['cross-provincial'] = cross_connections_df.copy()

connections_sp.dump(path='..')

unused_cnns_df.to_csv('unused_peer_connections.csv')

In [ ]:
# Let's view a few things about installed pipes
# avg age of the network from 2000 to 2025
plot_utils.plot_ages(all_connections_df=all_cnns_df).show()